# LLMy w Langgraph

W tym notebooku budujemy pipeline Langgraph z integracją modelu LLM przez **LangChain**.

Pipeline przyjmuje pytanie użytkownika, buduje prompt, wysyła go do modelu i zwraca odpowiedź.

Konfiguracja modelu (nazwa, temperatura, max tokeny, system prompt) przekazywana jest przez `configurable`.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Pipeline state

Stan pipeline'u przechowuje pytanie użytkownika, zbudowany prompt oraz odpowiedź modelu.

In [ ]:
from typing import TypedDict


class State(TypedDict):
    question: str
    prompt_messages: list[dict]
    llm_response: str
    formatted_output: str

## Node'y

Pipeline składa się z trzech node'ów:
1. **build_prompt** – buduje listę wiadomości (system + user) na podstawie pytania i konfiguracji
2. **call_llm** – wysyła prompt do modelu przez LangChain / OpenRouter
3. **format_output** – formatuje odpowiedź do czytelnej postaci

In [ ]:
from langchain_core.runnables import RunnableConfig


def build_prompt_node(state: State, config: RunnableConfig):
    configurable = config.get("configurable", {})
    system_prompt = configurable.get(
        "system_prompt",
        "Jesteś pomocnym asystentem. Odpowiadaj zwięźle i rzeczowo po polsku."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": state["question"]},
    ]
    return {"prompt_messages": messages}

In [ ]:
from langchain_openai import ChatOpenAI


def call_llm_node(state: State, config: RunnableConfig):
    configurable = config.get("configurable", {})
    model_name = configurable.get("model_name", "openai/gpt-4o-mini")
    temperature = configurable.get("temperature", 0.7)
    max_tokens = configurable.get("max_tokens", 512)

    llm = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    response = llm.invoke(state["prompt_messages"])
    return {"llm_response": response.content}

In [ ]:
def format_output_node(state: State, config: RunnableConfig):
    configurable = config.get("configurable", {})
    model_name = configurable.get("model_name", "openai/gpt-4o-mini")

    output = (
        f"=== Odpowiedź modelu ({model_name}) ===\n"
        f"Pytanie: {state['question']}\n"
        f"Odpowiedź:\n{state['llm_response']}"
    )
    return {"formatted_output": output}

## State graph

In [ ]:
from langgraph.graph import StateGraph, START, END


graph = StateGraph(State)

graph.add_node("build_prompt", build_prompt_node)
graph.add_node("call_llm", call_llm_node)
graph.add_node("format_output", format_output_node)

graph.add_edge(START, "build_prompt")
graph.add_edge("build_prompt", "call_llm")
graph.add_edge("call_llm", "format_output")
graph.add_edge("format_output", END)

pipeline = graph.compile()

In [ ]:
from IPython.display import Image, display

display(Image(pipeline.get_graph().draw_mermaid_png(), width=120))

## Uruchomienie pipeline'u

In [ ]:
initial_state = {
    "question": "Czym jest LangGraph i do czego się go używa?"
}

runtime_config = {
    "configurable": {
        "model_name": "openai/gpt-4o-mini",
        "temperature": 0.5,
        "max_tokens": 256,
        "system_prompt": "Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.",
    }
}

result = pipeline.invoke(initial_state, config=runtime_config)

In [ ]:
print(result["formatted_output"])

> **ZADANIA**